# Laboratório 02 — Qualidade e Preparação dos Dados

**Base:** Titanic (Seaborn)  
**Tempo estimado:** 50–70 minutos

## Objetivos
- localizar valores ausentes e duplicidades;
- selecionar estratégias de tratamento;
- converter tipos;
- codificar atributos categóricos;
- aplicar escalonamento.

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler

## 1. Carregamento

In [3]:
df = sns.load_dataset("titanic")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 2. Diagnóstico de qualidade

In [4]:
diagnostico = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "percentual_ausente": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique()
}).sort_values("percentual_ausente", ascending=False)

diagnostico

,tipo,ausentes,percentual_ausente,unicos
deck,category,688,77.22,7
age,float64,177,19.87,88
embarked,object,2,0.22,3
embark_town,object,2,0.22,3
sex,object,0,0.00,2
pclass,int64,0,0.00,3
survived,int64,0,0.00,2
fare,float64,0,0.00,248
parch,int64,0,0.00,7
sibsp,int64,0,0.00,7


In [5]:
print("Duplicidades:", df.duplicated().sum())

Duplicidades: 107


### Atividade 1
Identifique:
- as três colunas com maior percentual de ausência;
- uma coluna que pode ser removida;
- uma coluna que deve ser tratada, mas preservada.

**Resposta:**

- as três colunas com maior percentual de ausência são: deck,age, embarked
- uma coluna que pode ser removida é a coluna deck
- uma coluna que deve ser tratada, mas preservada é a coluna age

## 3. Cópia para preparação

In [6]:
dados = df.copy()

# Remoção de colunas com muita redundância ou ausência
colunas_remover = ["deck", "alive", "class", "who", "adult_male", "embark_town"]
dados = dados.drop(columns=colunas_remover)

# Tratamento de ausências
dados["age"] = dados["age"].fillna(dados["age"].median())
dados["embarked"] = dados["embarked"].fillna(dados["embarked"].mode()[0])
dados["fare"] = dados["fare"].fillna(dados["fare"].median())

# Remoção de duplicidades
dados = dados.drop_duplicates()

dados.isna().sum()

,0
survived,0
pclass,0
sex,0
age,0
sibsp,0
parch,0
fare,0
embarked,0
alone,0


## 4. Codificação de variáveis categóricas

In [7]:
dados_codificados = pd.get_dummies(
    dados,
    columns=["sex", "embarked"],
    drop_first=True,
    dtype=int
)

dados_codificados.head()

,survived,pclass,age,sibsp,parch,fare,alone,sex_male,embarked_Q,embarked_S
0,0,3,22.0,1,0,7.2500,False,1,0,1
1,1,1,38.0,1,0,71.2833,False,0,0,0
2,1,3,26.0,0,0,7.9250,True,0,0,1
3,1,1,35.0,1,0,53.1000,False,0,0,1
4,0,3,35.0,0,0,8.0500,True,1,0,1


## 5. Escalonamento

In [8]:
colunas_escalar = ["age", "fare", "sibsp", "parch"]

scaler = StandardScaler()
dados_padronizados = dados_codificados.copy()
dados_padronizados[colunas_escalar] = scaler.fit_transform(
    dados_padronizados[colunas_escalar]
)

dados_padronizados[colunas_escalar].describe().T

,count,mean,std,min,25%,50%,75%,max
age,775.0,2.521281e-16,1.000646,-2.119661,-0.623747,-0.114933,0.466569,3.664831
fare,775.0,-1.134576e-16,1.000646,-0.665941,-0.512240,-0.362359,-0.012993,9.116066
sibsp,775.0,-3.208903e-17,1.000646,-0.534545,-0.534545,-0.534545,0.475876,7.548821
parch,775.0,1.375244e-17,1.000646,-0.500754,-0.500754,-0.500754,0.689689,6.641909


### Atividade 2
Explique:
1. Por que a mediana foi utilizada em `age`?
2. Qual a finalidade de `get_dummies`?
3. O que mudou após o `StandardScaler`?

**Resposta:**

1. A variável age possui uma distribuição que pode conter valores extremos e assimetria. A mediana é uma medida de tendência central robusta que não é distorcida por esses valores extremos, ao contrário da média. Além disso, imputar pela mediana preserva o centro da distribuição de forma segura para o preenchimento dos valores nulos.
2. A finalidade de get_dummies é converter variáveis categóricas (como texto, rótulos ou categorias não numéricas) em variáveis indicadoras (dummy/one-hot encoding), que são representadas numericamente por 0 e 1.
3. As variáveis contínuas (age, fare, sibsp, parch) foram transformadas para uma mesma escala.

## 6. Comparação antes e depois

In [9]:
comparacao = pd.DataFrame({
    "media_antes": dados_codificados[colunas_escalar].mean(),
    "desvio_antes": dados_codificados[colunas_escalar].std(),
    "media_depois": dados_padronizados[colunas_escalar].mean(),
    "desvio_depois": dados_padronizados[colunas_escalar].std()
}).round(3)

comparacao

,media_antes,desvio_antes,media_depois,desvio_depois
age,29.581,13.766,0.0,1.001
fare,34.878,52.408,-0.0,1.001
sibsp,0.529,0.990,-0.0,1.001
parch,0.421,0.841,0.0,1.001


## Entrega
Salve como `LAB02_NomeSobrenome.ipynb` e registre todas as justificativas.